In [ ]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root))
from experiments.paths import GENERATED, PROJECT_ROOT

run_directory = GENERATED / 'soay/size_only/inference'
run_directory.mkdir(parents=True, exist_ok=True)
os.chdir(run_directory)


# Posterior Soay Sheep IPMs

Construct one GLM IPM and one GP IPM for every posterior draw, then save population growth, stable size distributions, reproductive values, and reproductive stable-size distributions.

In [ ]:
from pathlib import Path; import sys; sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
import tensorflow as tf
import gpflow
from scipy.special import expit
from gpflow.conditionals.util import base_conditional_with_lm
from experiments.soay.common.ipm_utils import calculate_ipm_metrics, make_gp_ipm, mk_K_glm, predict_y_mean, reproduction_fn_glm

In [ ]:
project_dir = PROJECT_ROOT

In [ ]:
data = pd.read_csv(GENERATED / 'soay/size_only/data/gp_ibm_dataset.csv')
true_metrics = dict(np.load(GENERATED / 'soay/size_only/data/true_gp_metrics.npz'))
data.describe().round(2)

In [ ]:
L, U, n_mesh = 1.7, 3.4, 250
n = 5000

In [ ]:
with np.load(GENERATED / 'soay/size_only/inference/gpdf_glm_samples.npz') as saved:
    glm_samples = {name: saved[name] for name in saved.files}
with np.load(GENERATED / 'soay/size_only/inference/gpdf_gp_samples.npz') as saved:
    gp_samples = {name: saved[name] for name in saved.files}


In [ ]:
recruit_int = np.mean(gp_samples['recr_int'])
print(recruit_int)

In [ ]:
gp_samples['recr_int'] = recruit_int
glm_samples['recr_int'] = recruit_int

In [ ]:
abc_deeprv_file = GENERATED / 'soay/size_only/abc/deeprv/abc_samples.npz'
abc_file = GENERATED / 'soay/size_only/abc/gp/abc_samples.npz'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

# Load original weighted ABC particles
with np.load(abc_deeprv_file) as abc:
    deeprv_indices = abc["indices"]
    deeprv_weights = abc["weights"]

with np.load(abc_file) as abc:
    gp_indices = abc["indices"]
    gp_weights = abc["weights"]

# Normalise weights
deeprv_weights = deeprv_weights / deeprv_weights.sum()
gp_weights = gp_weights / gp_weights.sum()

rate_columns = {
    "growth": 0,
    "surv": 1,
    "repr": 2,
    "rcsz": 3,
}

# Collect scalar parameters
plot_params = []

for rate, col in rate_columns.items():
    for name, values in gp_samples.items():
        if name.startswith(f"{rate}_") and np.asarray(values).ndim == 1:
            plot_params.append((rate, col, name))

# Grid dimensions
ncols = 4
nrows = math.ceil(len(plot_params) / ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(4 * ncols, 3 * nrows)
)

axes = np.asarray(axes).ravel()

for ax, (rate, col, parameter_name) in zip(axes, plot_params):

    gp_values = gp_samples[parameter_name][gp_indices[:, col]]
    deeprv_values = gp_samples[parameter_name][deeprv_indices[:, col]]

    # Identical bins for both posteriors
    bins = np.histogram_bin_edges(
        np.concatenate([gp_values, deeprv_values]),
        bins=50
    )

    ax.hist(
        gp_values,
        bins=bins,
        weights=gp_weights,
        density=True,
        alpha=0.5,
        label="GP ABC"
    )

    ax.hist(
        deeprv_values,
        bins=bins,
        weights=deeprv_weights,
        density=True,
        alpha=0.5,
        label="DeepRV ABC"
    )

    ax.set_title(parameter_name)
    ax.set_ylabel("Density")

# Remove unused axes
for ax in axes[len(plot_params):]:
    ax.remove()

# One legend for entire figure
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=2,
    bbox_to_anchor=(0.5, 1.01)
)

plt.tight_layout()
plt.show()

In [ ]:
def sample_ABC(filepath, true_samples, n_samples, rng_seed=123):
    with np.load(filepath) as abc:
        indices = abc['indices']
        weights = abc['weights']
        threshold = abc['threshold']
    rng = np.random.default_rng(rng_seed)
    particle_rows = rng.choice(len(indices),
                           size=n_samples,
                           replace=True,
                           p=weights)
    sampled_indices = indices[particle_rows]
    rate_columns ={
        'growth': 0,
        'surv': 1,
        'repr': 2,
        'rcsz': 3
        }   
    samples = {}

    for rate, col in rate_columns.items():
        rate_indices = sampled_indices[:, col]
        for parameter_name in true_samples.keys():
            if parameter_name.startswith(f'{rate}_'):
                samples[parameter_name] = true_samples[parameter_name][rate_indices]

    samples['recr_int'] = recruit_int
    return samples


In [ ]:
abc_deeprv_file = Path(GENERATED / 'soay/size_only/abc/deeprv/abc_samples.npz')
abc_deeprv_samples = sample_ABC(abc_deeprv_file, gp_samples, n_samples=5000, rng_seed=123)

In [ ]:
abc_file = Path(GENERATED / 'soay/size_only/abc/gp/abc_samples.npz')
abc_samples = sample_ABC(abc_file, gp_samples, n_samples=5000, rng_seed=123)

In [ ]:
X_surv = data[["z"]].to_numpy(np.float64)
X_repr = data[["z", "Repr"]].dropna()[["z"]].to_numpy(np.float64)
X_growth = data[["z", "z1"]].dropna()[["z"]].to_numpy(np.float64)
X_rcsz = data[["z", "Rcsz"]].dropna()[["z"]].to_numpy(np.float64)


In [ ]:
def draw(samples, index): 
    return {name: value.item() if value.ndim == 0 else value[index] for name, value in samples.items()}

def allocate(n):
    return {"lambda": np.empty(n), 
            "omega": np.empty((n, n_mesh)), 
            "omega_rep": np.empty((n, n_mesh)), 
            "gamma": np.empty((n, n_mesh))}

def store(target, index, metrics):
    for name in target:
        target[name][index] = metrics[name]

In [ ]:
class MCMCConditionedGP:
    def __init__(self, x_train, mu_train, variance, lengthscale, intercept, likelihood, jitter=5e-4):
        dtype = gpflow.default_float()
        self.x_train = tf.convert_to_tensor(x_train, dtype=dtype)
        self.intercept = tf.convert_to_tensor(intercept, dtype=dtype)
        self.likelihood = likelihood
        centered = np.asarray(mu_train, dtype=np.float64).reshape(-1, 1) - float(intercept)
        self.f_train_centered = tf.convert_to_tensor(centered, dtype=dtype)
        self.kernel = gpflow.kernels.SquaredExponential(variance=float(variance), lengthscales=float(lengthscale))
        n = int(self.x_train.shape[0])
        covariance = self.kernel(self.x_train, self.x_train) + tf.cast(jitter, dtype) * tf.eye(n, dtype=dtype)
        self.cholesky = tf.linalg.cholesky(covariance)

    def _x(self, x):
        return tf.reshape(tf.convert_to_tensor(x, dtype=gpflow.default_float()), (-1, 1))

    def predict_f(self, x, full_cov=False, full_output_cov=False):
        x = self._x(x)
        cross_covariance = self.kernel(self.x_train, x)
        new_covariance = self.kernel(x, x) if full_cov else self.kernel.K_diag(x)
        mean, variance = base_conditional_with_lm(cross_covariance, #Standard Gaussian Conditioning, q_sqrt = None
                                                  self.cholesky, 
                                                  new_covariance, 
                                                  self.f_train_centered, 
                                                  full_cov=full_cov, 
                                                  white=False)
        return mean + self.intercept, variance

    def predict_y(self, x, full_cov=False, full_output_cov=False):
        x = self._x(x)
        mean, variance = self.predict_f(x)
        return self.likelihood.predict_mean_and_var(x, mean, variance)

def make_model(rate, x_train, params, likelihood):
    return MCMCConditionedGP(x_train, params[f"{rate}_mu"], params[f"{rate}_var"], params[f"{rate}_ls"], params[f"{rate}_int"], likelihood)

In [ ]:
import matplotlib.pyplot as plt
training_inputs = {"surv": X_surv, "repr": X_repr, "growth": X_growth, "rcsz": X_rcsz}

def evaluate_ipm_models(model_samples, true_metrics, plot=False, epsilon=1e-12):
    def normalise(x):
        x = np.clip(np.asarray(x), epsilon, None)
        return x / x.sum(axis=-1, keepdims=True)

    true_lambda = float(true_metrics["lambda"])
    true_omega = normalise(true_metrics["omega"])
    true_gamma = np.asarray(true_metrics["gamma"])
    true_omega_rep = normalise(true_metrics["omega_rep"])
    scores = {}

    for model, samples in model_samples.items():
        lambdas = np.asarray(samples["lambda"])
        omegas = normalise(samples["omega"])
        gammas = np.asarray(samples["gamma"])
        omega_reps = normalise(samples["omega_rep"])

        lambda_errors = lambdas - true_lambda
        lambda_bias = lambda_errors.mean()
        lambda_mse = np.mean(lambda_errors ** 2)
        lambda_variance = lambdas.var()
        omega_kl = np.sum(true_omega * np.log(true_omega / omegas), axis=1)
        gamma_cosine = np.sum(gammas * true_gamma, axis=1) / (np.linalg.norm(gammas, axis=1) * np.linalg.norm(true_gamma))
        omega_rep_kl = np.sum(true_omega_rep * np.log(true_omega_rep / omega_reps), axis=1)

        scores[model] = {"Lambda MSE": lambda_mse, "Lambda Bias Squared": lambda_bias ** 2, "Lambda Variance": lambda_variance, "Lambda Bias": lambda_bias, "Stable State KL": omega_kl, "Reproductive Value Cosine Similarity": gamma_cosine, "Reproductive Stable Distribution KL": omega_rep_kl}

    if plot:
        plt.style.use("seaborn-v0_8-white")

        colors = {
            "GLM": "#4C78A8",
            "GP": "#E07B39",
            "ABC GP": "#59A14F",
            "ABC DeepRV": "#B07AA1",
        }

        panels = [
            ("Stable State KL", "Stable State Distribution", "KL Divergence"),
            ("Reproductive Value Cosine Similarity", "Reproductive Value", "Cosine Similarity"),
            ("Reproductive Stable Distribution KL", "Reproductive Stable Distribution", "KL Divergence"),
        ]

        fig, axes = plt.subplots(1, 3, figsize=(10.8, 3.4), constrained_layout=True)

        for model in scores:
            for axis, (metric, title, xlabel) in zip(axes, panels):
                pd.Series(scores[model][metric]).plot.density(
                    ax=axis,
                    color=colors[model],
                    linewidth=1.7,
                    alpha=0.9,
                    label=model,
                )

        axes[0].set_xlim(left=0)
        axes[2].set_xlim(left=0)

        cosine_limits = axes[1].get_xlim()
        axes[1].set_xlim(max(-1, cosine_limits[0]), min(1, cosine_limits[1]))

        for axis, (_, title, xlabel) in zip(axes, panels):
            axis.set_title(title, fontsize=11)
            axis.set_xlabel(xlabel, fontsize=10)
            axis.set_ylabel("")
            axis.tick_params(labelsize=9, direction="out")
            axis.grid(False)
            axis.spines["top"].set_visible(False)
            axis.spines["right"].set_visible(False)

        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(
            handles,
            labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 1.10),
            ncol=len(scores),
            frameon=False,
            fontsize=9,
        )
        fig.supylabel("Density", fontsize=11)

        fig.savefig("ipm_model_comparison.pdf", bbox_inches="tight")
        fig.savefig("ipm_model_comparison.png", dpi=300, bbox_inches="tight")
        plt.show()

    return pd.DataFrame({model: {metric: np.mean(values) for metric, values in model_scores.items()} for model, model_scores in scores.items()}).T

In [ ]:
from scipy.stats import norm
import matplotlib.pyplot as plt

n = 5000
mesh = L + (np.arange(n_mesh) + 0.5) * (U - L) / n_mesh
h = (U - L) / n_mesh
rates = ["surv", "repr", "growth", "rcsz"]
recruitment_int = gp_samples["recr_int"].mean()

def glm_predictions(p):
    return {
        "surv": expit(p["surv_int"] + p["surv_slope"] * mesh),
        "repr": expit(p["repr_int"] + p["repr_slope"] * mesh),
        "growth": p["growth_int"] + p["growth_slope"] * mesh,
        "growth_var": np.full(n_mesh, p["growth_noise"] ** 2),
        "rcsz": p["rcsz_int"] + p["rcsz_slope"] * mesh,
        "rcsz_var": np.full(n_mesh, p["rcsz_noise"] ** 2),
    }

def gp_predictions(p):
    predictions = {}
    for rate in rates:
        if rate in ["surv", "repr"]:
            likelihood = gpflow.likelihoods.Bernoulli(invlink=tf.sigmoid)
        else:
            likelihood = gpflow.likelihoods.Gaussian(variance=p[f"{rate}_noise"] ** 2)

        model = make_model(rate, training_inputs[rate], p, likelihood)
        mean, variance = model.predict_y(mesh)

        predictions[rate] = mean.numpy().ravel()

        if rate in ["growth", "rcsz"]:
            predictions[f"{rate}_var"] = variance.numpy().ravel()

    return predictions

def make_kernel(v, p):
    survival = v["surv"]
    reproduction = v["repr"]
    recruitment = expit(p["recr_int"])

    growth = norm.pdf(
        mesh[:, None],
        loc=v["growth"][None, :],
        scale=np.sqrt(v["growth_var"])[None, :],
    )

    recruit_size = norm.pdf(
        mesh[:, None],
        loc=v["rcsz"][None, :],
        scale=np.sqrt(v["rcsz_var"])[None, :],
    )

    P = h * survival[None, :] * growth
    fecundity = survival * reproduction * p["female_prob"] * recruitment
    F = h * fecundity[None, :] * recruit_size

    P *= survival / P.sum(axis=0)
    F *= fecundity / F.sum(axis=0)

    return P + F

methods = {
    "GLM": (glm_samples, glm_predictions),
    "GP": (gp_samples, gp_predictions),
    "ABC GP": (abc_samples, gp_predictions),
    "ABC DeepRV": (abc_deeprv_samples, gp_predictions)
}

prediction_cache_file = GENERATED / 'soay/size_only/inference/make_ipm_deeprv_predictions.npz'
# Set this to True to refresh the cache after changing prediction or kernel code.
recompute_predictions = False
cache_input_files = [GENERATED / 'soay/size_only/inference/gpdf_glm_samples.npz', GENERATED / 'soay/size_only/inference/gpdf_gp_samples.npz', abc_file, abc_deeprv_file]
cache_metadata = {
    "cache_version": np.array(1),
    "n": np.array(n),
    "mesh": mesh,
    "method_names": np.array(list(methods)),
    "rate_names": np.array(rates),
    "input_paths": np.array([str(path.resolve()) for path in cache_input_files]),
    "input_mtimes": np.array([path.stat().st_mtime_ns for path in cache_input_files]),
    "input_sizes": np.array([path.stat().st_size for path in cache_input_files]),
}
data_keys = [f"vital_rate_{method_index}_{rate}" for method_index in range(len(methods)) for rate in rates] + [f"ipm_metric_{method_index}_{metric}" for method_index in range(len(methods)) for metric in allocate(1)]
cache_loaded = False

if prediction_cache_file.exists() and not recompute_predictions:
    try:
        with np.load(prediction_cache_file) as saved:
            cache_loaded = all(name in saved and np.array_equal(saved[name], value) for name, value in cache_metadata.items()) and all(name in saved for name in data_keys)
            if cache_loaded:
                vital_rate_draws = {method: {rate: saved[f"vital_rate_{method_index}_{rate}"] for rate in rates} for method_index, method in enumerate(methods)}
                ipm_metrics = {method: {metric: saved[f"ipm_metric_{method_index}_{metric}"] for metric in allocate(1)} for method_index, method in enumerate(methods)}
    except (OSError, ValueError):
        cache_loaded = False

if cache_loaded:
    print(f"Loaded Predictions From {prediction_cache_file}")
else:
    vital_rate_draws = {}
    ipm_metrics = {}

    for method, (samples, prediction_function) in methods.items():
        vital_rate_draws[method] = {rate: np.empty((n, n_mesh)) for rate in rates}
        ipm_metrics[method] = allocate(n)

        for i in range(n):
            params = draw(samples, i)
            params["female_prob"] = 0.5
            params["recr_int"] = recruitment_int
            predictions = prediction_function(params)

            for rate in rates:
                vital_rate_draws[method][rate][i] = predictions[rate]

            K = make_kernel(predictions, params)
            metrics = calculate_ipm_metrics(K, predictions["repr"])
            store(ipm_metrics[method], i, metrics)

        print(f"{method} Complete")

    cached_arrays = cache_metadata.copy()
    for method_index, method in enumerate(methods):
        cached_arrays.update({f"vital_rate_{method_index}_{rate}": values for rate, values in vital_rate_draws[method].items()})
        cached_arrays.update({f"ipm_metric_{method_index}_{metric}": values for metric, values in ipm_metrics[method].items()})
    np.savez_compressed(prediction_cache_file, **cached_arrays)
    print(f"Saved Predictions To {prediction_cache_file}")

In [ ]:
plot_details = {
    "surv": ("Survival Vital Rate", "Survival Probability", "Surv"),
    "repr": ("Reproduction Vital Rate", "Reproduction Probability", "Repr"),
    "growth": ("Growth Vital Rate", "Expected Size Next Year", "z1"),
    "rcsz": ("Recruit Size Vital Rate", "Expected Recruit Size", "Rcsz"),
}

colors = {
    "GLM": "#0072B2",         # Blue
    "GP": "#D55E00",          # Vermillion
    "ABC GP": "#009E73",      # Green
    "ABC DeepRV": "#CC79A7",  # Purple
}

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)

for ax, rate in zip(axes.flat, rates):
    
    title, ylabel, data_column = plot_details[rate]
    observed = data[["z", data_column]].dropna()

    ax.scatter(
        observed["z"],
        observed[data_column],
        color="black",
        s=14,
        alpha=0.2,
        label="True Data",
    )

    for method, color in colors.items():
        values = vital_rate_draws[method][rate]
        mean = values.mean(axis=0)
        lower, upper = np.quantile(values, [0.025, 0.975], axis=0)

        ax.plot(mesh, mean, color=color, label=method)
        ax.fill_between(mesh, lower, upper, color=color, alpha=0.15)

    ax.set_title(title)
    ax.set_xlabel("Current Size")
    ax.set_ylabel(ylabel)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.style.use("seaborn-v0_8-white")

plot_details = {
    "surv": ("Survival Vital Rate", "Survival Probability", "Surv"),
    "repr": ("Reproduction Vital Rate", "Reproduction Probability", "Repr"),
    "growth": ("Growth Vital Rate", "Expected Size Next Year", "z1"),
    "rcsz": ("Recruit Size Vital Rate", "Expected Recruit Size", "Rcsz"),
}

colors = {
    "GLM": "#0072B2",
    "GP": "#D55E00",
    "ABC GP": "#009E73",
    "ABC DeepRV": "#CC79A7",
}

line_styles = {
    "GLM": "-",
    "GP": "--",
    "ABC GP": "-.",
    "ABC DeepRV": ":",
}

model_order = [method for method in methods if method in colors]

fig, axes = plt.subplots(2, 2, figsize=(12, 8.5), sharex=True, constrained_layout=True)

for axis, rate in zip(axes.flat, rates):
    title, ylabel, data_column = plot_details[rate]
    observed = data[["z", data_column]].dropna()

    axis.scatter(observed["z"], observed[data_column], color="0.2", s=16, alpha=0.25, edgecolors="none", label="Observed Data", zorder=1)

    for method in model_order:
        values = vital_rate_draws[method][rate]
        mean = values.mean(axis=0)
        lower, upper = np.quantile(values, [0.025, 0.975], axis=0)
        axis.fill_between(mesh, lower, upper, color=colors[method], alpha=0.1, linewidth=0, zorder=2)
        axis.plot(mesh, mean, color=colors[method], linestyle=line_styles[method], linewidth=2, alpha=0.9, label=method, zorder=3)

    axis.set_title(title, fontsize=13)
    axis.set_ylabel(ylabel, fontsize=11)
    axis.grid(False)
    axis.tick_params(labelsize=10, direction="out")
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

fig.supxlabel("Current Size", fontsize=14)
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.04), ncol=len(labels), frameon=False, fontsize=10)

fig.savefig("vital_rate_comparison.pdf", bbox_inches="tight")
fig.savefig("vital_rate_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.style.use("seaborn-v0_8-white")

colors = {
    "GLM": "#4C78A8",         # Blue
    "GP": "#E07B39",          # Orange
    "ABC GP": "#59A14F",      # Green
    "ABC DeepRV": "#B07AA1",  # Purple
}

all_lambdas = np.concatenate([ipm_metrics[method]["lambda"] for method in methods])
bins = np.linspace(all_lambdas.min(), all_lambdas.max(), 50)

fig, axis = plt.subplots(figsize=(7.2, 5))

for method in methods:
    axis.hist(
        ipm_metrics[method]["lambda"],
        bins=bins,
        density=True,
        color=colors[method],
        linewidth=1.8,
        alpha=0.7,
        label=method,
    )

axis.axvline(
    float(true_metrics["lambda"]),
    color="black",
    linestyle="--",
    linewidth=1.8,
    label=r"True $\lambda$",
)

axis.set_xlabel(r"Population Growth Rate $\lambda$", fontsize=11)
axis.set_ylabel("Density", fontsize=11)
axis.set_title("Posterior Distribution Of Population Growth Rate", fontsize=12)
axis.tick_params(labelsize=9, direction="out")
axis.grid(False)
axis.spines["top"].set_visible(False)
axis.spines["right"].set_visible(False)
axis.legend(frameon=False, fontsize=9, ncol=2)

fig.tight_layout()
fig.savefig("population_growth_rate.pdf", bbox_inches="tight")
fig.savefig("population_growth_rate.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
results_table = evaluate_ipm_models(
    ipm_metrics,
    true_metrics,
    plot=True,
)

In [ ]:
display(results_table)

np.savez_compressed("posterior_ipm_metrics.npz", glm_lambda=glm_metrics["lambda"], glm_omega=glm_metrics["omega"], glm_omega_rep=glm_metrics["omega_rep"], glm_gamma=glm_metrics["gamma"], gp_lambda=gp_metrics["lambda"], gp_omega=gp_metrics["omega"], gp_omega_rep=gp_metrics["omega_rep"], gp_gamma=gp_metrics["gamma"])
summary = pd.DataFrame({"GLM Mean": [glm_metrics["lambda"].mean()], "GLM SD": [glm_metrics["lambda"].std()], "GP Mean": [gp_metrics["lambda"].mean()], "GP SD": [gp_metrics["lambda"].std()], "True": [float(true_metrics["lambda"])]}, index=["Population Growth Rate"])
summary